In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# Step 1: Define the Euclidean Distance Function
def euclidean_distance(vectors):
    (feat_vecA, feat_vecB) = vectors
    sum_squared = tf.reduce_sum(tf.square(feat_vecA - feat_vecB), axis=1, keepdims=True)
    return tf.sqrt(tf.maximum(sum_squared, tf.keras.backend.epsilon()))

# Step 2: Create a Simple Convolutional Neural Network (CNN) for Feature Extraction
def build_siamese_cnn(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (3, 3), activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(128, activation="relu")(x)
    return Model(inputs, outputs)

# Step 3: Define the Siamese Network Architecture
def build_siamese_network(input_shape):
    # Shared CNN for feature extraction
    cnn = build_siamese_cnn(input_shape)

    # Inputs for the two images
    inputA = layers.Input(shape=input_shape)
    inputB = layers.Input(shape=input_shape)

    # Extract features using the shared CNN
    feat_vecA = cnn(inputA)
    feat_vecB = cnn(inputB)

    # Compute the Euclidean Distance between the feature vectors
    distance = layers.Lambda(euclidean_distance)([feat_vecA, feat_vecB])

    # Output Layer: Predict similarity (binary output: 0 or 1)
    outputs = layers.Dense(1, activation="sigmoid")(distance)

    return Model(inputs=[inputA, inputB], outputs=outputs)

# Step 4: Compile the Siamese Network
input_shape = (105, 105, 1)  # Example input shape for grayscale images
siamese_model = build_siamese_network(input_shape)
siamese_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
siamese_model.summary()

# Step 5: Prepare Dummy Training Data
# Simulated pairs of inputs (e.g., signature images) and labels (1: same, 0: different)
num_samples = 1000
X1 = np.random.rand(num_samples, 105, 105, 1)  # First set of images
X2 = np.random.rand(num_samples, 105, 105, 1)  # Second set of images
y = np.random.randint(0, 2, num_samples)       # Labels indicating similarity

# Step 6: Train the Siamese Network
siamese_model.fit([X1, X2], y, batch_size=32, epochs=10)

# Step 7: Evaluate the Model
# Simulated test data
X1_test = np.random.rand(100, 105, 105, 1)
X2_test = np.random.rand(100, 105, 105, 1)
y_test = np.random.randint(0, 2, 100)

# Evaluate the model on test data
loss, accuracy = siamese_model.evaluate([X1_test, X2_test], y_test)
print(f"\nTest Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")
